Krzysztof Jastrzębski
149292

# Lab 2: Lab 2: Apache Spark — analiza danych transakcyjnych
## Część 1
### SparkSession, wczytanie danych, transformacja danych

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")


Spark 4.0.0-preview2 — gotowy


In [2]:
df = spark.read.json("transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()

df.show(10, truncate=False)

Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)

+------+-----------+--------+-------------------+-------+-------+
|amount|category   |store   |timestamp          |tx_id  |user_id|
+------+-----------+--------+-------------------+-------+-------+
|312.32|elektronika|Warszawa|2026-04-12 08:25:07|TX00001|u48    |
|79.57 |książki    |Warszawa|2026-04-12 08:05:43|TX00002|u15    |
|126.17|odzież     |Warszawa|2026-04-12 09:15:30|TX00003|u18    |
|34.08 |odzież     |Warszawa|2026-04-12 10:05:39|TX00004|u10    |
|428.88|żywność    |Kraków  |2026-04-12 09:04:36|TX00005|u17    |
|345.21|książki    |Warszawa|2026-04-12 09:36:31|TX00006|u25    |
|376.42|żywność    |Warszawa|2026-04-12 10:06:49|TX00007|u15    |
|85.36 |elektronika|Gdańsk  |2026-04-12 09:08:25|TX00008|u24    |
|66.26 |żywno

In [3]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()  # timestamp powinien być teraz 'timestamp (nullable = true)'

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



## Część 2 - agregacje
### Zadanie 2.1 - Liczba transakcji i suma przychodów per sklep

In [4]:
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("store")
)
store_summary.show()

+--------+---------+----------+-----------+
|   store|liczba_tx|  suma_PLN|srednia_PLN|
+--------+---------+----------+-----------+
|  Gdańsk|     2498|1021266.35|     408.83|
|  Kraków|     2522|1025896.95|     406.78|
|Warszawa|     2424| 961642.24|     396.72|
| Wrocław|     2556|1002739.21|     392.31|
+--------+---------+----------+-----------+



### Zadanie 2.2 - Statystyki per kategoria

In [5]:
from pyspark.sql.functions import min as _min, max as _max

# TWÓJ KOD
# df.groupBy("category").agg(...).orderBy("category").show()
category_summary = (
    df.groupBy("category")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("category")
).show()

+-----------+---------+----------+-----------+
|   category|liczba_tx|  suma_PLN|srednia_PLN|
+-----------+---------+----------+-----------+
|elektronika|     2542|1520770.69|     598.26|
|    książki|     2574| 851382.08|     330.76|
|     odzież|     2453| 849877.55|     346.46|
|    żywność|     2431| 789514.43|     324.77|
+-----------+---------+----------+-----------+



## Część 3 - tumbling windows
### Zadanie 3.1 - Liczba transakcji per godzina (tumbling 1h)

In [7]:
from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"))    # okno 1-godzinne
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)
hourly.show(truncate=False)

+------------------------------------------+---------+----------+
|window                                    |liczba_tx|suma_PLN  |
+------------------------------------------+---------+----------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|3150     |1241911.3 |
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|4661     |1896230.21|
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|2189     |873403.24 |
+------------------------------------------+---------+----------+



In [8]:
(
    hourly
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .show(truncate=False)
)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
+-------------------+-------------------+---------+----------+



### Zadanie 3.2 - Okna 30-minutowe per sklep

In [9]:
halfhourly = (
    df.groupBy(window("timestamp", "30 minutes"), "store")    # okno 1-godzinne
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)
halfhourly.show(truncate=False)

+------------------------------------------+--------+---------+---------+
|window                                    |store   |liczba_tx|suma_PLN |
+------------------------------------------+--------+---------+---------+
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Wrocław |296      |111540.59|
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Gdańsk  |252      |93391.22 |
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Kraków  |289      |117786.42|
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Warszawa|275      |88441.58 |
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Gdańsk  |514      |209187.85|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Wrocław |502      |215587.17|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Kraków  |532      |223541.41|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Warszawa|490      |182435.06|
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|Kraków  |590      |224358.03|
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|Warszawa|584      |214573.66|
|{2026-04-12 09:00:00, 2026-04-12 09:3

### Zadanie 3.3 - W której godzinie sklep “Kraków” miał najwyższy przychód?

In [12]:
hourly_krakow_max = (
    df.filter("store = \"Kraków\"")
    .groupBy(window("timestamp", "1 hour"))# okno 1-godzinne
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("suma_PLN", ascending = False)
)
hourly_krakow_max.show(n=1, truncate=False)

+------------------------------------------+---------+---------+
|window                                    |liczba_tx|suma_PLN |
+------------------------------------------+---------+---------+
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|1169     |483309.86|
+------------------------------------------+---------+---------+
only showing top 1 row



## Część 4 - sliding windows
### Zadanie 4.1 - Okno 1h, krok 30 minut

In [19]:
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))  # szerokość 1h, krok 30min
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN")
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od")
)
sliding.show(truncate=False)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 07:30:00|2026-04-12 08:30:00|1112     |411159.81 |
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 08:30:00|2026-04-12 09:30:00|4443     |1753033.6 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 09:30:00|2026-04-12 10:30:00|3696     |1557641.39|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
|2026-04-12 10:30:00|2026-04-12 11:30:00|749      |289709.95 |
+-------------------+-------------------+---------+----------+



### Zadanie 4.2 - tumbling vs sliding

In [15]:
tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)
sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)
print(f"Tumbling (1h):          {tumbling_rows} okien")
print(f"Sliding  (1h / 30min):  {sliding_rows} okien")

# Odpowiedz w komentarzu: dlaczego sliding ma więcej wierszy?
# TWOJA ODPOWIEDŹ: poniewaz okna sliding rozpoczynaja sie co 30 minut i nakladaja sie na siebie

Tumbling (1h):          3 okien
Sliding  (1h / 30min):  7 okien


## Część 5

In [16]:
# Odpowiedz na pytania w komentarzach:

# 1. Ile transakcji jest w oknie 09:00–10:00?
#    Sprawdź w wyniku zadania 3.1.
#    ODPOWIEDŹ: 4661

# 2. Jaka jest różnica między groupBy("store") a groupBy(window(...), "store")?
#    ODPOWIEDŹ: groupBy("store") podzieli transakcje na grupy unikalne po jednym kluczu (tylko po sklepie), 
#               groupBy(window(...), "store") dodatkowo uwzględnia okno czasowe dla danego sklepu

# 3. W oknie sliding 1h/30min — ile okien zawiera transakcje z godziny 09:30?
#    Wskazówka: narysuj oś czasu.
#    ODPOWIEDŹ: w sparku przedziały dla okien sa domkniete w czasie startu i otwarte w czasie konca, wiec transakcje 
#               z 9:30 pojawia sie w 2 oknach - [9:00 - 10:00) i [9:30 - 10:30) - nie bedzie ich w oknie [8:30 - 9:30)

## Praca domowa
### Zadanie 1

In [17]:
(
    df.filter("store = 'Gdańsk'")
    .groupBy(window("timestamp", "1 hour"))
    .agg(
        _round(avg("amount"), 2).alias("srednia_PLN"),
        count("tx_id").alias("liczba_tx"),
    )
    .orderBy("srednia_PLN")
    .show(truncate=False)
)

+------------------------------------------+-----------+---------+
|window                                    |srednia_PLN|liczba_tx|
+------------------------------------------+-----------+---------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|395.01     |766      |
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|412.92     |558      |
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|415.91     |1174     |
+------------------------------------------+-----------+---------+



### Zadanie 2

In [18]:
(
    df.groupBy(window("timestamp", "30 minutes"), "category")
    .agg(count("tx_id").alias("liczba_tx"))
    .filter("window.start = '2026-04-12 09:00:00'")
    .orderBy("category")
    .show(truncate=False)
)

+------------------------------------------+-----------+---------+
|window                                    |category   |liczba_tx|
+------------------------------------------+-----------+---------+
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|elektronika|611      |
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|książki    |622      |
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|odzież     |605      |
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|żywność    |567      |
+------------------------------------------+-----------+---------+



### Zadanie 3

In [20]:
# Zadanie 3: 15-minutowe okna — szczyt transakcji (łącznie dla wszystkich sklepów)
(
    df.groupBy(window("timestamp", "15 minutes"))
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("liczba_tx", ascending = False)
    .show(truncate=False)
)

+------------------------------------------+---------+---------+
|window                                    |liczba_tx|suma_PLN |
+------------------------------------------+---------+---------+
|{2026-04-12 09:15:00, 2026-04-12 09:30:00}|1234     |481566.97|
|{2026-04-12 09:00:00, 2026-04-12 09:15:00}|1171     |440715.14|
|{2026-04-12 09:30:00, 2026-04-12 09:45:00}|1156     |504943.74|
|{2026-04-12 08:45:00, 2026-04-12 09:00:00}|1139     |475251.18|
|{2026-04-12 09:45:00, 2026-04-12 10:00:00}|1100     |469004.36|
|{2026-04-12 08:30:00, 2026-04-12 08:45:00}|899      |355500.31|
|{2026-04-12 10:00:00, 2026-04-12 10:15:00}|858      |359254.89|
|{2026-04-12 08:15:00, 2026-04-12 08:30:00}|644      |213061.19|
|{2026-04-12 10:15:00, 2026-04-12 10:30:00}|582      |224438.4 |
|{2026-04-12 08:00:00, 2026-04-12 08:15:00}|468      |198098.62|
|{2026-04-12 10:30:00, 2026-04-12 10:45:00}|443      |156900.51|
|{2026-04-12 10:45:00, 2026-04-12 11:00:00}|306      |132809.44|
+------------------------